# Hopfield 1982 复现实验
## 存储容量与信噪比

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Heptazero/nn-labs/blob/main/hopfield-1982/hopfield_1982_capacity.ipynb)

对应论文：J. J. Hopfield, *Neural networks and physical systems with emergent collective computational abilities* (1982)。

这份 notebook 不是把代码切成很多格子，而是把一次复现实验组织成可以检查的推理链：先说清问题和预期，再实现最小模型，最后比较理论与仿真。选择 **Runtime → Run all** 即可从头运行。

## 0. 实验地图

| 环节 | 本实验要回答什么 |
|---|---|
| 论文问题 | 一个 $N=100$ 的 Hopfield 网络能稳定存储多少条随机记忆？ |
| 操纵变量 | 存储记忆数 $n$ |
| 控制变量 | 神经元数 $N$、更新规则、随机种子、试验次数 |
| 观测指标 | 平均错误 bit 数、完全召回率、5 bit 内近似召回率 |
| 理论预期 | $n$ 增大时串扰噪声增大，在 $n\approx0.15N$ 附近明显退化 |
| 复现标准 | 趋势和临界区域接近论文图 2，不要求有限样本数值逐点一致 |

## 1. 论文问题与数学预期

### 1.1 为什么记忆越多，召回越容易出错？

Hebb 权重把所有记忆叠加在同一个矩阵中。检索某一条目标记忆时，目标自身产生“信号”，其他记忆产生“串扰噪声”。论文给出的近似量级为：

$$
\text{signal}\approx\frac{N}{2},\qquad
\sigma_{\text{noise}}\approx\sqrt{\frac{(n-1)N}{2}}.
$$

因此 $n$ 增大时，信号基本不变，噪声却按 $\sqrt{n-1}$ 增长。单个 bit 的理论错误概率近似为：

$$
P_{\mathrm{bit}}
=\frac12\operatorname{erfc}\left(
\frac{N/2}{\sqrt{2}\,\sigma_{\text{noise}}}
\right).
$$

若暂时把各 bit 是否出错视为独立事件，则整条 $N$ bit 记忆完全正确的粗略概率是 $(1-P_{\mathrm{bit}})^N$。这里的独立性只是近似，所以理论曲线主要用来检查量级和趋势。

### 1.2 本实验采用哪一种状态编码？

这里忠实采用论文的 $V_i\in\{0,1\}$ 状态，同时用 $\mu_i=2V_i-1\in\{-1,+1\}$ 构造 Hebb 权重：

$$
T_{ij}=\sum_{s=1}^{n}\mu_i^s\mu_j^s,\qquad T_{ii}=0.
$$

这样可以直接对应论文中信号约为 $N/2$ 的推导。早期实现曾使用 $\pm1$ 状态；两种编码表达的是同一吸引子机制，但理论常数不能不加说明地混用。

## 2. 环境与实验配置

Colab 已预装 NumPy 和 Matplotlib，本实验不需要 GPU，也不需要在本地创建虚拟环境。所有会影响结果的参数集中在一个配置对象中，避免散落在不同单元格。

In [ ]:
from dataclasses import dataclass
from math import erfc, sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


@dataclass(frozen=True)
class ExperimentConfig:
    N: int = 100
    n_min: int = 1
    n_max: int = 20
    trials_per_n: int = 100
    max_sweeps: int = 50
    near_error_bits: int = 5
    seed: int = 0


CONFIG = ExperimentConfig()
CONFIG

## 3. 构造记忆与 Hebb 权重

### 3.1 随机记忆

每条记忆是长度为 $N$ 的独立随机二进制向量。随机模式不是现实数据的完整模型，而是容量理论的基线：先把结构化相关性排除，单独观察存储数量造成的串扰。

### 3.2 权重矩阵

先把 $0/1$ 状态变换为零均值的 $\pm1$ 编码，再对所有记忆做外积求和。对角线清零表示神经元不与自身连接。

In [ ]:
def make_memories(n: int, N: int, rng: np.random.Generator) -> np.ndarray:
    """生成形状为 (n, N) 的 0/1 随机记忆。"""
    return rng.integers(0, 2, size=(n, N), dtype=np.int8)


def hebb_weights(memories: np.ndarray) -> np.ndarray:
    """按论文式 (2) 构造对称 Hebb 权重，并清除自连接。"""
    centered = 2 * memories.astype(np.int32) - 1
    weights = centered.T @ centered
    np.fill_diagonal(weights, 0)
    return weights


# 最小自检：权重必须对称且没有自连接。
_rng = np.random.default_rng(CONFIG.seed)
_sample_memories = make_memories(n=5, N=CONFIG.N, rng=_rng)
_sample_weights = hebb_weights(_sample_memories)
assert np.array_equal(_sample_weights, _sample_weights.T)
assert np.all(np.diag(_sample_weights) == 0)
print("权重矩阵自检通过：对称，且 T_ii = 0")

## 4. 异步更新与收敛

### 4.1 为什么必须异步？

异步更新一次只更新一个神经元，并立刻把新状态交给下一个神经元。对于对称权重，每次有效翻转都不会提高能量，因此系统最终到达不动点。同步更新则可能在两个状态之间往返，本实验不采用同步更新。

### 4.2 更新规则

神经元 $i$ 的局部场为 $h_i=\sum_{j\ne i}T_{ij}V_j$。当 $h_i>0$ 时取 1，当 $h_i<0$ 时取 0；恰好等于 0 时保持原状态，避免人为规定平局方向。

In [ ]:
def energy(weights: np.ndarray, state: np.ndarray) -> float:
    """对称网络的能量，只用于检查单调性。"""
    return float(-0.5 * state @ weights @ state)


def run_async(
    weights: np.ndarray,
    initial_state: np.ndarray,
    rng: np.random.Generator,
    max_sweeps: int,
    track_energy: bool = False,
):
    """随机顺序逐个更新神经元，直到一整轮没有状态变化。"""
    state = initial_state.copy()
    energy_trace = [energy(weights, state)] if track_energy else None

    for sweep in range(max_sweeps):
        changed = False
        for i in rng.permutation(state.size):
            local_field = weights[i] @ state
            new_value = 1 if local_field > 0 else 0 if local_field < 0 else state[i]
            if new_value != state[i]:
                state[i] = new_value
                changed = True
                if track_energy:
                    energy_trace.append(energy(weights, state))
        if not changed:
            return state, sweep + 1, True, energy_trace

    return state, max_sweeps, False, energy_trace

### 4.3 先做机制自检，再跑大实验

在扫描容量前，先随机挑一个小例子，检查异步更新过程中能量是否单调不增。这个自检比“代码能运行”更重要：它验证实现是否保留了论文最核心的动力学性质。

In [ ]:
_rng = np.random.default_rng(CONFIG.seed + 1)
_memories = make_memories(n=5, N=CONFIG.N, rng=_rng)
_weights = hebb_weights(_memories)
_initial = _rng.integers(0, 2, size=CONFIG.N, dtype=np.int8)
_final, _sweeps, _converged, _energy_trace = run_async(
    _weights,
    _initial,
    _rng,
    max_sweeps=CONFIG.max_sweeps,
    track_energy=True,
)

assert _converged, "机制自检未在最大轮数内收敛"
assert np.all(np.diff(_energy_trace) <= 1e-9), "能量出现上升，更新实现需要检查"
print(f"机制自检通过：{_sweeps} sweep 收敛，{len(_energy_trace) - 1} 次有效翻转，能量单调不增")

## 5. 容量扫描设计

### 5.1 每个 $n$ 如何测量？

对于每个存储数 $n$：

1. 重新随机生成 $n$ 条记忆；
2. 构造对应权重矩阵；
3. 随机选择其中一条记忆作为初态；
4. 执行异步更新直到收敛；
5. 统计最终状态与目标记忆相差多少 bit；
6. 重复多次，降低偶然性。

从记忆本身出发测的是“已分配记忆是不是稳定点”。吸引盆实验会故意先翻转若干 bit，那是另一项实验。

In [ ]:
def theoretical_bit_error_probability(n: int, N: int) -> float:
    """论文信噪比近似下的单 bit 错误概率。"""
    if n <= 1:
        return 0.0
    sigma = sqrt((n - 1) * N / 2)
    return 0.5 * erfc((N / 2) / (sigma * sqrt(2)))


def run_capacity_sweep(config: ExperimentConfig):
    rng = np.random.default_rng(config.seed)
    rows = []

    for n in range(config.n_min, config.n_max + 1):
        errors = []
        converged_count = 0

        for _ in range(config.trials_per_n):
            memories = make_memories(n=n, N=config.N, rng=rng)
            weights = hebb_weights(memories)
            target = memories[rng.integers(n)].copy()
            recalled, _, converged, _ = run_async(
                weights,
                target,
                rng,
                max_sweeps=config.max_sweeps,
            )
            errors.append(int(np.count_nonzero(recalled != target)))
            converged_count += int(converged)

        errors = np.asarray(errors)
        bit_error_p = theoretical_bit_error_probability(n=n, N=config.N)
        rows.append(
            {
                "n": n,
                "load": n / config.N,
                "mean_errors": float(errors.mean()),
                "exact_recall_rate": float(np.mean(errors == 0)),
                "near_recall_rate": float(np.mean(errors <= config.near_error_bits)),
                "convergence_rate": converged_count / config.trials_per_n,
                "theory_bit_error_p": bit_error_p,
                "theory_exact_rate": (1 - bit_error_p) ** config.N,
            }
        )

    return rows

## 6. 一键运行实验

默认设置为 $N=100$、$n=1\ldots20$、每个 $n$ 重复 100 次。免费 Colab 的 CPU 足够运行。若只是检查 notebook 是否正常，可以先把 `trials_per_n` 改成 10；正式记录结果时再改回 100 或更高。

In [ ]:
results = run_capacity_sweep(CONFIG)

header = f"{'n':>3} {'n/N':>6} {'平均错位':>8} {'完全召回':>9} {'≤5位召回':>9} {'理论完全召回':>12}"
print(header)
print("-" * len(header))
for row in results:
    print(
        f"{row['n']:>3d} "
        f"{row['load']:>6.2f} "
        f"{row['mean_errors']:>8.2f} "
        f"{row['exact_recall_rate']:>9.2f} "
        f"{row['near_recall_rate']:>9.2f} "
        f"{row['theory_exact_rate']:>12.2f}"
    )

## 7. 可视化与读图

左轴画平均错误 bit 数，右轴画完全召回率和 5 bit 内近似召回率。读图时不要只找某个精确阈值，而要看三个问题：

1. 误差从哪个 $n$ 开始系统性上升？
2. 完全召回率何时明显下降？
3. 完全召回已经下降时，5 bit 内召回是否仍然保持较高？

第三点区分了“严格稳定性丢失”和“仍具有近似纠错能力”。

In [ ]:
n_values = np.array([row["n"] for row in results])
mean_errors = np.array([row["mean_errors"] for row in results])
exact_rates = np.array([row["exact_recall_rate"] for row in results])
near_rates = np.array([row["near_recall_rate"] for row in results])
theory_exact_rates = np.array([row["theory_exact_rate"] for row in results])

fig, ax_error = plt.subplots(figsize=(9, 5.5))
ax_error.plot(n_values, mean_errors, "o-", color="tab:red", label="Mean bit errors")
ax_error.set_xlabel("Number of stored memories n")
ax_error.set_ylabel("Mean bit errors", color="tab:red")
ax_error.tick_params(axis="y", labelcolor="tab:red")
ax_error.axvline(0.15 * CONFIG.N, color="0.5", linestyle="--", label="0.15N reference")

ax_rate = ax_error.twinx()
ax_rate.plot(n_values, exact_rates, "s-", label="Exact recall rate")
ax_rate.plot(n_values, near_rates, "^-", label=f"Recall within {CONFIG.near_error_bits} bits")
ax_rate.plot(n_values, theory_exact_rates, ":", color="tab:green", label="SNR approximation")
ax_rate.set_ylabel("Recall rate")
ax_rate.set_ylim(-0.03, 1.03)

handles_1, labels_1 = ax_error.get_legend_handles_labels()
handles_2, labels_2 = ax_rate.get_legend_handles_labels()
ax_error.legend(handles_1 + handles_2, labels_1 + labels_2, loc="center left")
ax_error.set_title(f"Hopfield 1982 storage-capacity reproduction (N={CONFIG.N})")
fig.tight_layout()

runtime_figure_path = Path("capacity_curve_colab.png")
fig.savefig(runtime_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"本次运行的图片已暂存为：{runtime_figure_path.resolve()}")

## 8. 与论文结果比较

论文报告的典型现象是：

- $n=5$ 时，分配记忆几乎全部稳定；
- $n\approx15=0.15N$ 时开始出现明显过载；
- 一部分记忆仍能落入少量 bit 误差内，另一部分会偏离很远；
- 信噪比近似能解释退化趋势，但有限网络中的 bit 错误并不真正独立，因此理论完全召回率不会与仿真逐点重合。

运行后应根据上图自己填写：

1. 本次完全召回率开始明显下降的 $n$：`待填写`；
2. 本次平均错误数明显上升的 $n$：`待填写`；
3. 是否在 $n\approx15$ 附近看到论文描述的退化：`待填写`；
4. 理论近似与仿真偏差最大的区间：`待填写`。

不要把 notebook 中预写的论文结论当成自己的实验结论；只有运行后观察到的结果才进入实验记录。

## 9. 保存图片：临时输出与长期档案

上一步已经把图片写入 Colab 临时目录。运行时断开后，`/content/` 会被清空。若要长期保存，可以把下面的开关改为 `True`，图片会复制到 Google Drive；默认保持 `False`，这样执行 **Run all** 时不会弹出 Drive 授权。

最终建议：大量中间结果放 Drive，确认值得引用的小图再提交到 GitHub 的 `hopfield-1982/figures/`，或者复制进 Obsidian 的论文档案包。

In [ ]:
SAVE_TO_GOOGLE_DRIVE = False

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_figure_dir = Path("/content/drive/MyDrive/nn-labs/hopfield-1982/figures")
    drive_figure_dir.mkdir(parents=True, exist_ok=True)
    destination = drive_figure_dir / runtime_figure_path.name
    destination.write_bytes(runtime_figure_path.read_bytes())
    print(f"图片已保存到：{destination}")
else:
    print("本次未写入 Google Drive；如需长期保存，请把 SAVE_TO_GOOGLE_DRIVE 改为 True。")

## 10. 自检与下一步

### 10.1 理解自检

1. 为什么增加 $n$ 不会增强目标信号，却会增大串扰噪声？
2. 为什么从已存记忆本身出发只能测稳定性，不能测吸引盆半径？
3. 为什么理论单 bit 错误率接近仿真，并不保证整条记忆的完全召回率也精确匹配？

### 10.2 代码自检

- 权重矩阵是否对称，且 $T_{ii}=0$？
- 异步更新过程中能量是否单调不增？
- 改变随机种子后，趋势是否仍然存在？
- 增加试验次数后，曲线是否更平滑？

### 10.3 下一项实验

在确认本 notebook 的结构和讲解密度适合自己之后，下一份复现可以直接复制这十个标题，把操纵变量改成“初态与目标记忆的汉明距离”，测量吸引盆范围与纠错能力。